# HealthKit Pipeline — Post-Deploy Constraints

Run this notebook **after** the HealthKit SDP pipeline has completed its
first successful update. Applies metadata across all pipeline layers:

1. **Column comments** — human-readable descriptions on every column
   across bronze_typed, silver (SCD1), and gold tables
2. **Primary key RELY constraints** — `uuid` / `stage_uuid` on silver SCD1,
   `record_id` on bronze_typed tables, composite `(user_id, activity_date)`
   on silver_activity_summaries
3. **Foreign key RELY constraints** — `bronze_typed_*.record_id` →
   `wearables_zerobus.record_id`

All constraints are informational (`RELY` / `NOT ENFORCED`) — they inform the
query optimizer and document data relationships without runtime overhead.

**Usage:** Run manually after first pipeline deploy, or schedule as a
post-deploy job task.

**Tables covered:** 5 bronze_typed, 4 silver SCD1, 1 gold (14 tables total
including SCD2 and quarantine, but only SCD1 tables get PK constraints).

In [0]:
dbutils.widgets.text("catalog", "hls_fde_dev", "Catalog")
dbutils.widgets.text("schema", "dev_matthew_giglia_wearables", "Schema")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
fqn = f"{catalog}.{schema}"
print(f"Applying constraints to: {fqn}")

In [0]:
%sql
ALTER TABLE ${catalog}.${schema}.silver_health_samples
  ALTER COLUMN ingested_at COMMENT 'Server-side ingestion timestamp from ZeroBus (AUTO CDC sequence key)',
  ALTER COLUMN user_id COMMENT 'Authenticated user ID from JWT claims (joins to auth.users)',
  ALTER COLUMN source_platform COMMENT 'Data source platform (apple_healthkit, synthetic, etc.)',
  ALTER COLUMN uuid COMMENT 'HealthKit sample UUID — primary key, CDC dedup key',
  ALTER COLUMN sample_type COMMENT 'HKQuantityTypeIdentifier (e.g. HeartRate, StepCount, OxygenSaturation)',
  ALTER COLUMN value COMMENT 'Measured numeric value in the units specified by the unit column',
  ALTER COLUMN unit COMMENT 'Unit of measurement (count/min, count, kcal, %, ms, m, etc.)',
  ALTER COLUMN start_ts COMMENT 'Sample measurement start timestamp (UTC)',
  ALTER COLUMN end_ts COMMENT 'Sample measurement end timestamp (UTC)',
  ALTER COLUMN source_name COMMENT 'Device or app that produced the sample (e.g. Apple Watch)',
  ALTER COLUMN source_bundle_id COMMENT 'Bundle identifier of the source app',
  ALTER COLUMN metadata COMMENT 'Optional HealthKit metadata dictionary (VARIANT, shredded)',
  ALTER COLUMN sample_date COMMENT 'Date extracted from start_ts for liquid clustering',
  ALTER COLUMN sample_hour COMMENT 'Hour-of-day (0–23) for intraday pattern analysis'

In [0]:
%sql
ALTER TABLE ${catalog}.${schema}.silver_workouts
  ALTER COLUMN ingested_at COMMENT 'Server-side ingestion timestamp from ZeroBus (AUTO CDC sequence key)',
  ALTER COLUMN user_id COMMENT 'Authenticated user ID from JWT claims',
  ALTER COLUMN source_platform COMMENT 'Data source platform identifier',
  ALTER COLUMN uuid COMMENT 'HealthKit workout UUID — primary key, CDC dedup key',
  ALTER COLUMN activity_type COMMENT 'Human-readable activity type (walking, cycling, running, etc.)',
  ALTER COLUMN activity_type_raw COMMENT 'Apple HKWorkoutActivityType raw integer enum value',
  ALTER COLUMN duration_seconds COMMENT 'Total workout duration in seconds',
  ALTER COLUMN start_ts COMMENT 'Workout start timestamp (UTC)',
  ALTER COLUMN end_ts COMMENT 'Workout end timestamp (UTC)',
  ALTER COLUMN total_distance_meters COMMENT 'Total distance covered in meters (NULL if not applicable)',
  ALTER COLUMN total_energy_burned_kcal COMMENT 'Total active energy burned in kilocalories',
  ALTER COLUMN source_name COMMENT 'Device that recorded the workout',
  ALTER COLUMN metadata COMMENT 'Optional workout metadata dictionary (VARIANT)',
  ALTER COLUMN workout_date COMMENT 'Date extracted from start_ts for liquid clustering',
  ALTER COLUMN duration_minutes COMMENT 'Derived: duration_seconds / 60',
  ALTER COLUMN distance_km COMMENT 'Derived: total_distance_meters / 1000'

In [0]:
%sql
ALTER TABLE ${catalog}.${schema}.silver_sleep_stages
  ALTER COLUMN ingested_at COMMENT 'Server-side ingestion timestamp from ZeroBus (AUTO CDC sequence key)',
  ALTER COLUMN user_id COMMENT 'Authenticated user ID from JWT claims',
  ALTER COLUMN source_platform COMMENT 'Data source platform identifier',
  ALTER COLUMN stage_uuid COMMENT 'Per-stage HealthKit UUID — primary key, CDC dedup key (one per sleep stage)',
  ALTER COLUMN stage COMMENT 'Sleep stage type: asleepDeep, asleepREM, asleepCore, or awake',
  ALTER COLUMN stage_start_ts COMMENT 'Stage start timestamp (UTC)',
  ALTER COLUMN stage_end_ts COMMENT 'Stage end timestamp (UTC)',
  ALTER COLUMN stage_duration_minutes COMMENT 'Derived: (stage_end_ts - stage_start_ts) in minutes',
  ALTER COLUMN session_start_ts COMMENT 'Parent sleep session start (UTC, for grouping stages into sessions)',
  ALTER COLUMN session_end_ts COMMENT 'Parent sleep session end (UTC)',
  ALTER COLUMN sleep_date COMMENT 'Date of session onset for liquid clustering and daily aggregation'

In [0]:
%sql
ALTER TABLE ${catalog}.${schema}.silver_activity_summaries
  ALTER COLUMN ingested_at COMMENT 'Server-side ingestion timestamp from ZeroBus',
  ALTER COLUMN user_id COMMENT 'Authenticated user ID from JWT claims',
  ALTER COLUMN source_platform COMMENT 'Data source platform identifier',
  ALTER COLUMN activity_date COMMENT 'Calendar date for the activity summary (one per user per day)',
  ALTER COLUMN energy_burned_kcal COMMENT 'Active energy burned (Move ring value) in kilocalories',
  ALTER COLUMN energy_goal_kcal COMMENT 'Daily Move ring goal in kilocalories',
  ALTER COLUMN exercise_minutes COMMENT 'Exercise minutes logged (Exercise ring value)',
  ALTER COLUMN exercise_goal_minutes COMMENT 'Daily Exercise ring goal in minutes',
  ALTER COLUMN stand_hours COMMENT 'Stand hours completed (Stand ring value)',
  ALTER COLUMN stand_goal_hours COMMENT 'Daily Stand ring goal in hours',
  ALTER COLUMN energy_goal_pct COMMENT 'Derived: energy_burned / energy_goal (1.0 = Move ring closed)',
  ALTER COLUMN exercise_goal_pct COMMENT 'Derived: exercise_minutes / exercise_goal (1.0 = Exercise ring closed)',
  ALTER COLUMN stand_goal_pct COMMENT 'Derived: stand_hours / stand_goal (1.0 = Stand ring closed)'

In [0]:
%sql
-- bronze_typed_deletes
ALTER TABLE ${catalog}.${schema}.bronze_typed_deletes
  ALTER COLUMN record_id COMMENT 'Bronze record GUID — 1:1 with source ingestion row (PK)',
  ALTER COLUMN ingested_at COMMENT 'Server-side ingestion timestamp from ZeroBus',
  ALTER COLUMN user_id COMMENT 'Authenticated user ID of the user who deleted the sample',
  ALTER COLUMN source_platform COMMENT 'Data source platform identifier',
  ALTER COLUMN deleted_uuid COMMENT 'UUID of the deleted HealthKit record (targets silver tables via CDC)',
  ALTER COLUMN sample_type COMMENT 'HK type identifier — determines target table for delete routing';

-- gold_sleep_sessions
ALTER TABLE ${catalog}.${schema}.gold_sleep_sessions
  ALTER COLUMN user_id COMMENT 'Authenticated user ID from JWT claims',
  ALTER COLUMN source_platform COMMENT 'Data source platform identifier',
  ALTER COLUMN session_start_ts COMMENT 'Sleep session start (UTC, from HealthKit session boundary)',
  ALTER COLUMN session_end_ts COMMENT 'Sleep session end (UTC, from HealthKit session boundary)',
  ALTER COLUMN sleep_date COMMENT 'Date of session onset for liquid clustering',
  ALTER COLUMN deep_sleep_minutes COMMENT 'Total minutes in deep (slow-wave) sleep stage',
  ALTER COLUMN rem_sleep_minutes COMMENT 'Total minutes in REM sleep stage',
  ALTER COLUMN core_sleep_minutes COMMENT 'Total minutes in core (light) sleep stage',
  ALTER COLUMN awake_minutes COMMENT 'Total minutes spent awake within session window',
  ALTER COLUMN total_tracked_minutes COMMENT 'Sum of all stage durations (deep + REM + core + awake)',
  ALTER COLUMN total_sleep_minutes COMMENT 'Derived: deep + REM + core (excludes awake)',
  ALTER COLUMN session_duration_minutes COMMENT 'Wall-clock time: session_end_ts - session_start_ts in minutes',
  ALTER COLUMN sleep_efficiency COMMENT 'Derived: total_sleep_minutes / session_duration_minutes',
  ALTER COLUMN stage_count COMMENT 'Total number of stages in session',
  ALTER COLUMN deep_stage_count COMMENT 'Number of deep sleep stage transitions',
  ALTER COLUMN rem_stage_count COMMENT 'Number of REM stage transitions',
  ALTER COLUMN core_stage_count COMMENT 'Number of core sleep stage transitions',
  ALTER COLUMN awake_stage_count COMMENT 'Number of awake stage transitions'

In [0]:
%sql
-- =============================================================================
-- Primary Key constraints (RELY / NOT ENFORCED)
-- =============================================================================

-- Silver SCD1 tables: keyed on HealthKit UUID (deduplicated by AUTO CDC)
ALTER TABLE ${catalog}.${schema}.silver_health_samples
  ADD CONSTRAINT silver_health_samples_pk PRIMARY KEY (uuid) RELY;

ALTER TABLE ${catalog}.${schema}.silver_workouts
  ADD CONSTRAINT silver_workouts_pk PRIMARY KEY (uuid) RELY;

ALTER TABLE ${catalog}.${schema}.silver_sleep_stages
  ADD CONSTRAINT silver_sleep_stages_pk PRIMARY KEY (stage_uuid) RELY;

-- Activity summaries: one row per user per day (no UUID, no CDC)
ALTER TABLE ${catalog}.${schema}.silver_activity_summaries
  ADD CONSTRAINT silver_activity_summaries_pk PRIMARY KEY (user_id, activity_date) RELY;

-- Bronze typed tables: keyed on record_id (1:1 with source ingestion)
ALTER TABLE ${catalog}.${schema}.bronze_typed_health_samples
  ADD CONSTRAINT bronze_typed_health_samples_pk PRIMARY KEY (record_id) RELY;

ALTER TABLE ${catalog}.${schema}.bronze_typed_workouts
  ADD CONSTRAINT bronze_typed_workouts_pk PRIMARY KEY (record_id) RELY;

-- Note: bronze_typed_sleep_stages uses stage_uuid as PK (record_id is NOT unique —
-- one sleep session explodes to many stage rows)
ALTER TABLE ${catalog}.${schema}.bronze_typed_sleep_stages
  ADD CONSTRAINT bronze_typed_sleep_stages_pk PRIMARY KEY (stage_uuid) RELY;

ALTER TABLE ${catalog}.${schema}.bronze_typed_activity_summaries
  ADD CONSTRAINT bronze_typed_activity_summaries_pk PRIMARY KEY (record_id) RELY;

ALTER TABLE ${catalog}.${schema}.bronze_typed_deletes
  ADD CONSTRAINT bronze_typed_deletes_pk PRIMARY KEY (record_id) RELY;

In [0]:
%sql
-- =============================================================================
-- Foreign Key constraints (RELY / NOT ENFORCED)
-- =============================================================================

-- Bronze typed tables reference the raw bronze source row.
-- Note: bronze_typed_sleep_stages is excluded because record_id is NOT unique
-- (one session record explodes into many stage rows).

ALTER TABLE ${catalog}.${schema}.bronze_typed_health_samples
  ADD CONSTRAINT bronze_typed_health_samples_bronze_fk
  FOREIGN KEY (record_id) REFERENCES ${catalog}.${schema}.wearables_zerobus (record_id) RELY;

ALTER TABLE ${catalog}.${schema}.bronze_typed_workouts
  ADD CONSTRAINT bronze_typed_workouts_bronze_fk
  FOREIGN KEY (record_id) REFERENCES ${catalog}.${schema}.wearables_zerobus (record_id) RELY;

ALTER TABLE ${catalog}.${schema}.bronze_typed_activity_summaries
  ADD CONSTRAINT bronze_typed_activity_summaries_bronze_fk
  FOREIGN KEY (record_id) REFERENCES ${catalog}.${schema}.wearables_zerobus (record_id) RELY;

ALTER TABLE ${catalog}.${schema}.bronze_typed_deletes
  ADD CONSTRAINT bronze_typed_deletes_bronze_fk
  FOREIGN KEY (record_id) REFERENCES ${catalog}.${schema}.wearables_zerobus (record_id) RELY;

In [0]:
print("\n" + "="*60)
print("HealthKit Pipeline constraints applied successfully!")
print("="*60)
print(f"\nCatalog.Schema: {fqn}")
print(f"\nColumn comments applied to:")
for t in ["silver_health_samples", "silver_workouts", "silver_sleep_stages",
          "silver_activity_summaries", "bronze_typed_deletes", "gold_sleep_sessions"]:
    print(f"  - {fqn}.{t}")
print(f"\nPrimary Key (RELY) constraints:")
print(f"  - silver_health_samples (uuid)")
print(f"  - silver_workouts (uuid)")
print(f"  - silver_sleep_stages (stage_uuid)")
print(f"  - silver_activity_summaries (user_id, activity_date)")
print(f"  - bronze_typed_health_samples (record_id)")
print(f"  - bronze_typed_workouts (record_id)")
print(f"  - bronze_typed_sleep_stages (stage_uuid)")
print(f"  - bronze_typed_activity_summaries (record_id)")
print(f"  - bronze_typed_deletes (record_id)")
print(f"\nForeign Key (RELY) constraints:")
print(f"  - bronze_typed_health_samples.record_id -> wearables_zerobus.record_id")
print(f"  - bronze_typed_workouts.record_id -> wearables_zerobus.record_id")
print(f"  - bronze_typed_activity_summaries.record_id -> wearables_zerobus.record_id")
print(f"  - bronze_typed_deletes.record_id -> wearables_zerobus.record_id")
print(f"\nAll constraints are RELY / NOT ENFORCED (optimizer hints only).")